In [0]:
from datetime import datetime, timedelta
import random


# ============================================================
# CRM SOURCE — CUSTOMERS
# ============================================================

customers = [
    (1001, " John Smith ", "Nigeria", "john.smith@email.com"),
    (1002, "mary jones", "Ghana", "mary.jones@email.com"),
    (1003, "DAVID BROWN", "nigeria", "david.brown@email.com"),
    (1004, "Sarah Williams", " Kenya ", "sarah.w@email.com"),
    (1005, "Michael Johnson", "Nigeria", None),
    (1006, "Grace Okafor", "NIGERIA", "grace.o@email.com"),
    (1007, "Daniel Mensah", "Ghana", "daniel.m@email.com"),
    (1008, "James Wilson", "kenya", "james.w@email.com"),
    (1009, " Linda Adeyemi", "Nigeria ", "linda.a@email.com"),
    (1010, "Peter Anderson", "Ghana", "peter@email"),
    (1011, "Emma Taylor", "Nigeria", "emma.t@email.com"),
    (1012, "Samuel Brown", "KENYA", "samuel.b@email.com"),
    (1013, "Olivia Davis", "Nigeria", "olivia.d@email.com"),
    (1014, "Benjamin Clark ", "ghana", "ben.c@email.com"),
    (1015, "Sophia Lewis", " Nigeria ", "sophia.l@email.com"),
    (1016, "Isaac Mensah", "Ghana", "isaac.m@email.com"),
    (1017, "Amelia Moore", "Kenya", "amelia.m@email.com"),
    (1018, "Joseph Ade", "Nigeria", "joseph.a@email.com"),
    (1019, "Isabella King", "Ghana", "isabella.k@email.com"),
    (1020, "Thomas Wright", "Nigeria", "thomas.w@email.com")
]


# ============================================================
# ERP SOURCE — PRODUCTS
# ============================================================

products = [
    (2001, "Laptop", "Electronics", 850.00),
    (2002, "monitor", "electronics", 300.00),
    (2003, "Keyboard ", "Accessories", 75.00),
    (2004, "Mouse", " accessories ", 40.00),
    (2005, "Printer", "Office Equipment", None),
    (2006, "Desk", "Furniture", 450.00),
    (2007, "Office Chair", "furniture", 350.00),
    (2008, " Headset", "Accessories", 120.00),
    (2009, "Webcam", "ELECTRONICS", 150.00),
    (2010, "Tablet", "Electronics", 500.00),
    (2011, "Laptop Bag", "Accessories", 65.00),
    (2012, "USB Cable", "accessories", 15.00),
    (2013, "External Hard Drive", "Storage", 120.00),
    (2014, "SSD", "storage", 180.00),
    (2015, "Router", "Networking", None),
    (2016, "Switch", "networking", 220.00),
    (2017, "Projector", "Office Equipment", 700.00),
    (2018, "Desk Lamp", "Furniture", 45.00),
    (2019, "Phone", "Electronics", 600.00),
    (2020, "Printer Ink", "office equipment", 55.00)
]


# ============================================================
# ERP SOURCE — SALES ORDERS
# ============================================================

orders = [
    (3001, 1001, 2001, 1, "2026-08-20"),
    (3002, 1002, 2003, 2, "2026-08-20"),
    (3003, 1003, 2002, 1, "2026-08-21"),
    (3004, 1004, 2007, 1, "2026-08-21"),
    (3005, 1005, 2004, 3, "2026-08-21"),
    (3006, 1006, 2005, 1, "2026-08-22"),
    (3007, 1007, 2008, 2, "2026-08-22"),
    (3008, 1008, 2009, 1, "2026-08-23"),
    (3009, 1009, 2006, 1, "2026-08-23"),
    (3010, 1010, 2010, 2, "2026-08-24"),
    (3011, 1011, 2011, 1, "2026-08-24"),
    (3012, 1012, 2012, 4, "2026-08-25"),
    (3013, 1013, 2013, 1, "2026-08-25"),
    (3014, 1014, 2014, 2, "2026-08-25"),
    (3015, 1015, 2015, 1, "2026-08-26"),
    (3016, 1016, 2016, 1, "2026-08-26"),
    (3017, 1017, 2017, 2, "2026-08-26"),
    (3018, 1018, 2018, 3, "2026-08-26"),
    (3019, 1019, 2019, 1, "2026-08-26"),
    (3020, 1020, 2020, 2, None)
]

In [0]:
# ============================================================
# CREATE SOURCE DATAFRAMES
# ============================================================

customer_df = spark.createDataFrame(
    customers,
    ["CustomerID", "CustomerName", "Country", "Email"]
)

product_df = spark.createDataFrame(
    products,
    ["ProductID", "ProductName", "Category", "Price"]
)

orders_df = spark.createDataFrame(
    orders,
    ["OrderID", "CustomerID", "ProductID", "Quantity", "OrderDate"]
)

In [0]:
display(customer_df)
display(product_df)
display(orders_df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
import uuid


# ============================================
# 1. CREATE INGESTION RUN METADATA
# ============================================

batch_id = str(uuid.uuid4())
ingestion_timestamp = current_timestamp()

print(f"Batch ID: {batch_id}")

In [0]:
# ============================================
# 2. PREPARE CUSTOMERS FOR BRONZE
# ============================================

customer_bronze = (
    customer_df
    .withColumn("_ingestion_timestamp", ingestion_timestamp)
    .withColumn("_source", lit("CRM"))
    .withColumn("_batch_id", lit(batch_id))
)


# ============================================
# 3. PREPARE PRODUCTS FOR BRONZE
# ============================================

product_bronze = (
    product_df
    .withColumn("_ingestion_timestamp", ingestion_timestamp)
    .withColumn("_source", lit("ERP"))
    .withColumn("_batch_id", lit(batch_id))
)


# ============================================
# 4. PREPARE ORDERS FOR BRONZE
# ============================================

orders_bronze = (
    orders_df
    .withColumn("_ingestion_timestamp", ingestion_timestamp)
    .withColumn("_source", lit("ERP"))
    .withColumn("_batch_id", lit(batch_id))
)

In [0]:
display(customer_bronze)

In [0]:
customer_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_customers")

product_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_products")

orders_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_sales_orders")

print("Bronze ingestion completed successfully.")
print(f"Batch ID: {batch_id}")

In [0]:
print("Customers:", spark.table("bronze_customers").count())
print("Products:", spark.table("bronze_products").count())
print("Orders:", spark.table("bronze_sales_orders").count())

In [0]:
display(
    spark.table("bronze_customers")
    .select("_source")
    .distinct()
)

display(
    spark.table("bronze_sales_orders")
    .select("_source")
    .distinct()
)

In [0]:
display(
    spark.table("bronze_customers")
    .select(
        "_batch_id",
        "_source",
        "_ingestion_timestamp"
    )
    .distinct()
)

display(
    spark.table("bronze_products")
    .select(
        "_batch_id",
        "_source",
        "_ingestion_timestamp"
    )
    .distinct()
)

display(
    spark.table("bronze_sales_orders")
    .select(
        "_batch_id",
        "_source",
        "_ingestion_timestamp"
    )
    .distinct()
)